# 02 — Model Training Pipeline
**Emotion-Aware E-Commerce Chatbot | MSc Dissertation | Heriot-Watt University**

## How to use this notebook
1. Click **Run All** at the top — that's it
2. Wait ~60 minutes
3. Check the final accuracy in the last cell

> Do NOT restart kernel mid-way. Run all cells top to bottom in one go.

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import sys, os, time
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, random_split
from transformers import (AutoTokenizer, AutoModel,
                           AutoModelForSequenceClassification,
                           TrainingArguments, Trainer,
                           get_linear_schedule_with_warmup)
from sklearn.metrics import accuracy_score, f1_score
from data.dataset_loader import (
    load_phase1_sentiment, load_phase2_high_frustration,
    load_phase3_moderate_frustration, load_phase4_full,
    FrustrationDataset, SentimentDataset, TWCS_PATH
)
from config import SENTIMENT_MODEL, DEVICE

# ── Paths ─────────────────────────────────────────────────────────────────────
P1_SAVE = os.path.abspath('../models/saved/sentiment_model')
P2_SAVE = os.path.abspath('../models/saved/frustration_model')
os.makedirs(P1_SAVE, exist_ok=True)
os.makedirs(P2_SAVE, exist_ok=True)

# ── Hyperparameters ───────────────────────────────────────────────────────────
P1_EPOCHS     = 3
P1_BATCH      = 16
P1_LR         = 2e-5
FRUST_EPOCHS  = 3
FRUST_BATCH   = 16
FRUST_LR      = 2e-5
MAX_LEN       = 128

print(f'Device:      {DEVICE}')
print(f'Base model:  {SENTIMENT_MODEL}')
print(f'P1 save:     {P1_SAVE}')
print(f'P2 save:     {P2_SAVE}')
print('\n✅ Setup complete — starting training pipeline')

In [ ]:
# ── Cell 2: Phase 1 — Train Sentiment Classifier ──────────────────────────────
print('='*60)
print('PHASE 1 — Sentiment Classification')
print('='*60)

p1 = load_phase1_sentiment()
texts1  = [e['text']  for e in p1]
labels1 = [e['label'] for e in p1]

tokenizer_p1 = AutoTokenizer.from_pretrained(SENTIMENT_MODEL)
ds1 = SentimentDataset(texts1, labels1, tokenizer_p1, MAX_LEN)

val_size  = max(200, int(0.1 * len(ds1)))
tr1, vl1  = random_split(ds1, [len(ds1)-val_size, val_size],
                          generator=torch.Generator().manual_seed(42))
print(f'Train: {len(tr1):,} | Val: {len(vl1):,}')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': round(accuracy_score(labels, preds), 4),
        'f1_macro': round(f1_score(labels, preds, average='macro'), 4)
    }

model_p1 = AutoModelForSequenceClassification.from_pretrained(
    SENTIMENT_MODEL, num_labels=3, ignore_mismatched_sizes=True).to(DEVICE)

args_p1 = TrainingArguments(
    output_dir=P1_SAVE,
    num_train_epochs=P1_EPOCHS,
    per_device_train_batch_size=P1_BATCH,
    per_device_eval_batch_size=P1_BATCH,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    learning_rate=P1_LR,
    logging_steps=50,
    fp16=(DEVICE=='cuda'),
    report_to='none'
)

trainer_p1 = Trainer(
    model=model_p1, args=args_p1,
    train_dataset=tr1, eval_dataset=vl1,
    compute_metrics=compute_metrics
)

t0 = time.time()
trainer_p1.train()
trainer_p1.save_model(P1_SAVE)
tokenizer_p1.save_pretrained(P1_SAVE)

r1 = trainer_p1.evaluate()
print(f'\n✅ PHASE 1 COMPLETE ({(time.time()-t0)/60:.1f} min)')
print(f'   Accuracy: {r1["eval_accuracy"]:.4f}')
print(f'   F1 Macro: {r1["eval_f1_macro"]:.4f}')
print(f'   Saved to: {P1_SAVE}')

In [ ]:
# ── Cell 3: Build Frustration Regressor (loads FROM Phase 1) ──────────────────
print('='*60)
print('Building Frustration Regressor from Phase 1 checkpoint')
print('='*60)

class FrustrationRegressor(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base)
        h = self.encoder.config.hidden_size
        self.head = nn.Sequential(
            nn.Dropout(0.1), nn.Linear(h, 128),
            nn.ReLU(), nn.Linear(128, 1), nn.Sigmoid()
        )
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.head(out.last_hidden_state[:,0,:]).squeeze(-1)

# Always load from Phase 1 saved checkpoint
base = P1_SAVE if os.path.exists(os.path.join(P1_SAVE, 'config.json')) else SENTIMENT_MODEL
print(f'Loading from: {base}')

tok_frust   = AutoTokenizer.from_pretrained(base)
frust_model = FrustrationRegressor(base).to(DEVICE)

def run_phase(examples, phase_name, epochs):
    """Train frustration model on a set of examples."""
    if not examples:
        print(f'  Skipping {phase_name} — no examples')
        return [], [], []

    texts  = [e['text']              for e in examples]
    scores = [e['frustration_score'] for e in examples]

    dataset  = FrustrationDataset(texts, scores, tok_frust, MAX_LEN)
    val_n    = max(50, int(0.1 * len(dataset)))
    train_n  = len(dataset) - val_n

    if train_n <= 0:
        print(f'  ⚠️ Too few examples ({len(dataset)}) — skipping {phase_name}')
        return [], [], []

    tr, vl = random_split(dataset, [train_n, val_n],
                           generator=torch.Generator().manual_seed(42))
    tr_dl  = DataLoader(tr, batch_size=FRUST_BATCH, shuffle=True)
    vl_dl  = DataLoader(vl, batch_size=FRUST_BATCH)

    opt    = AdamW(frust_model.parameters(), lr=FRUST_LR, weight_decay=0.01)
    total  = len(tr_dl) * epochs
    sched  = get_linear_schedule_with_warmup(opt, total//10, total)
    loss_fn = nn.MSELoss()

    t_losses, v_losses, maes = [], [], []
    best_val = float('inf')

    print(f'\n{phase_name} — {len(examples):,} examples | {epochs} epochs')
    t0 = time.time()

    for epoch in range(1, epochs+1):
        frust_model.train()
        t_loss = 0
        for b in tr_dl:
            ids  = b['input_ids'].to(DEVICE)
            mask = b['attention_mask'].to(DEVICE)
            lbls = b['labels'].to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(frust_model(ids, mask), lbls)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(frust_model.parameters(), 1.0)
            opt.step(); sched.step()
            t_loss += loss.item()

        frust_model.eval()
        v_loss, ep_maes = 0, []
        with torch.no_grad():
            for b in vl_dl:
                ids  = b['input_ids'].to(DEVICE)
                mask = b['attention_mask'].to(DEVICE)
                lbls = b['labels'].to(DEVICE)
                preds = frust_model(ids, mask)
                v_loss += loss_fn(preds, lbls).item()
                ep_maes.extend(torch.abs(preds-lbls).cpu().tolist())

        avg_t = t_loss/len(tr_dl)
        avg_v = v_loss/len(vl_dl)
        avg_m = np.mean(ep_maes)
        t_losses.append(avg_t); v_losses.append(avg_v); maes.append(avg_m)
        print(f'  Epoch {epoch}/{epochs} | Train MSE: {avg_t:.4f} | Val MSE: {avg_v:.4f} | MAE: {avg_m:.4f}')

        if avg_v < best_val:
            best_val = avg_v
            torch.save(frust_model.state_dict(), f'{P2_SAVE}/best_model.pt')
            print(f'    ✓ Best saved (Val MSE: {best_val:.4f})')

    print(f'  Completed in {(time.time()-t0)/60:.1f} min')
    return t_losses, v_losses, maes

print('\n✅ Frustration model built — ready to train phases 2, 3, 4')

In [ ]:
# ── Cell 4: Phase 2 — High Frustration ────────────────────────────────────────
print('='*60)
print('PHASE 2 — High Frustration Training')
print('='*60)
p2 = load_phase2_high_frustration()
p2_t, p2_v, p2_m = run_phase(p2, 'Phase 2 — High Frustration', FRUST_EPOCHS)
print('\n✅ Phase 2 complete')

In [ ]:
# ── Cell 5: Phase 3 — Moderate Frustration ────────────────────────────────────
print('='*60)
print('PHASE 3 — Moderate Frustration Training')
print('='*60)
p3 = load_phase3_moderate_frustration()
p3_t, p3_v, p3_m = run_phase(p3, 'Phase 3 — Moderate Frustration', FRUST_EPOCHS)
print('\n✅ Phase 3 complete')

In [ ]:
# ── Cell 6: Phase 4 — Full Dataset ────────────────────────────────────────────
print('='*60)
print('PHASE 4 — Full Dataset Fine-Tuning')
print('='*60)
p4 = load_phase4_full()
p4_t, p4_v, p4_m = run_phase(p4, 'Phase 4 — Full Dataset', epochs=4)
tok_frust.save_pretrained(P2_SAVE)
print(f'\n✅ Phase 4 complete — model saved to {P2_SAVE}')

In [ ]:
# ── Cell 7: Training Curves ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Phase 1 curves
p1_logs  = trainer_p1.state.log_history
t_logs   = [x for x in p1_logs if 'loss' in x and 'eval_loss' not in x]
e_logs   = [x for x in p1_logs if 'eval_loss' in x]
if t_logs:
    axes[0][0].plot([x['step'] for x in t_logs], [x['loss'] for x in t_logs],
                    color='#1565C0', linewidth=2)
axes[0][0].set_title('Phase 1 — Sentiment Training Loss', fontweight='bold')
axes[0][0].set_xlabel('Step'); axes[0][0].set_ylabel('Loss'); axes[0][0].grid(alpha=0.3)

if e_logs:
    axes[0][1].plot([x['epoch'] for x in e_logs], [x.get('eval_accuracy',0) for x in e_logs],
                    'o-', color='#1B5E20', linewidth=2, label='Accuracy')
    axes[0][1].plot([x['epoch'] for x in e_logs], [x.get('eval_f1_macro',0) for x in e_logs],
                    's-', color='#BF360C', linewidth=2, label='F1 Macro')
axes[0][1].set_title('Phase 1 — Validation Metrics', fontweight='bold')
axes[0][1].set_xlabel('Epoch'); axes[0][1].set_ylim(0,1)
axes[0][1].legend(); axes[0][1].grid(alpha=0.3)

# Frustration MSE curves
for (t_l, v_l, label, color) in [
    (p2_t, p2_v, 'Phase 2', '#ef5350'),
    (p3_t, p3_v, 'Phase 3', '#ff9800'),
    (p4_t, p4_v, 'Phase 4', '#7B1FA2'),
]:
    if t_l:
        axes[1][0].plot(range(1,len(t_l)+1), t_l, 'o-', color=color, linewidth=2, label=f'{label} Train')
        axes[1][0].plot(range(1,len(v_l)+1), v_l, 's--', color=color, linewidth=1.5, alpha=0.6)
axes[1][0].set_title('Frustration — MSE Loss All Phases', fontweight='bold')
axes[1][0].set_xlabel('Epoch'); axes[1][0].set_ylabel('MSE')
axes[1][0].legend(); axes[1][0].grid(alpha=0.3)

# MAE curves
for (m, label, color) in [(p2_m,'Phase 2','#ef5350'),(p3_m,'Phase 3','#ff9800'),(p4_m,'Phase 4','#7B1FA2')]:
    if m:
        axes[1][1].plot(range(1,len(m)+1), m, 'o-', color=color, linewidth=2, label=label)
axes[1][1].set_title('Frustration — MAE All Phases', fontweight='bold')
axes[1][1].set_xlabel('Epoch'); axes[1][1].set_ylabel('MAE')
axes[1][1].legend(); axes[1][1].grid(alpha=0.3)

plt.suptitle('Full Training Pipeline — All Phases', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../evaluation/results/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Charts saved to evaluation/results/training_curves.png')

In [ ]:
# ── Cell 8: Final Test ─────────────────────────────────────────────────────────
print('='*60)
print('FINAL MODEL TEST')
print('='*60)

test_messages = [
    ('hi how are you',                                        0.10),
    ('where is my order',                                     0.28),
    ('my order is late',                                      0.40),
    ('I have been waiting for 2 weeks and no one responds',   0.60),
    ('this is completely unacceptable I want a refund NOW',   0.80),
    ('WORST SERVICE EVER I will never use this again!!!',     0.92),
]

frust_model.eval()
print(f'\n{"Message":<52} {"Expected":>10} {"Predicted":>10} {"Diff":>8}')
print('-' * 84)

total_mae = 0
with torch.no_grad():
    for msg, expected in test_messages:
        enc  = tok_frust(msg, return_tensors='pt', truncation=True,
                         padding=True, max_length=MAX_LEN)
        pred = frust_model(
            enc['input_ids'].to(DEVICE),
            enc['attention_mask'].to(DEVICE)
        ).item()
        diff = pred - expected
        total_mae += abs(diff)
        status = '✅' if abs(diff) < 0.15 else '⚠️'
        print(f'{status} {msg[:50]:<50} {expected:>10.2f} {pred:>10.2f} {diff:>+8.2f}')

avg_mae = total_mae / len(test_messages)
print(f'\n{"─"*84}')
print(f'Average MAE on test messages: {avg_mae:.4f}')

if avg_mae < 0.10:
    print('🎉 Excellent accuracy!')
elif avg_mae < 0.15:
    print('✅ Good accuracy — suitable for dissertation')
elif avg_mae < 0.20:
    print('⚠️ Acceptable — consider more epochs for improvement')
else:
    print('❌ Low accuracy — check training completed correctly')

print(f'\nPhase 1 Accuracy: {r1["eval_accuracy"]:.4f}')
print(f'Phase 1 F1 Macro: {r1["eval_f1_macro"]:.4f}')
print(f'\n✅ Training complete — run streamlit run app.py to use trained model')